### Packages installation & files upload

In [8]:
!pip install Pillow
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tempfile
import os
import torch
import gc

In [9]:
from google.colab import files

uploaded = files.upload()

Saving pictures.zip to pictures.zip


In [10]:
import zipfile

zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("/content")

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### Data preprocessing

In [12]:
def set_image_dpi(file_path):

    im = Image.open(file_path)

    length_x, width_y = im.size

    factor = min(1, float(1024.0 / length_x))

    size = int(factor * length_x), int(factor * width_y)

    im_resized = im.resize(size, Image.Resampling.LANCZOS)

    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.png')

    temp_filename = temp_file.name

    im_resized.save(temp_filename, dpi=(300, 300))

    return temp_filename


def get_grayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

def remove_noise(image):
    return cv2.fastNlMeansDenoising(image, None, 10, 7, 21)

def normalize(img):
    norm_img = np.zeros((img.shape[0], img.shape[1]))
    img = cv2.normalize(img, norm_img, 0, 255, cv2.NORM_MINMAX)
    return img

def thin(img):
    bg = cv2.medianBlur(img, 51)
    norm = cv2.divide(img, bg, scale=255)
    binary = cv2.adaptiveThreshold(
        norm,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        41,
        10
    )

    kernel = np.ones((1, 1), np.uint8)
    clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    clean_inv = 255 - clean
    return clean_inv

def preprocess(img):
    img = np.array(img)
    img = normalize(img)
    img = get_grayscale(img)
    img = remove_noise(img)
    img = thin(img)
    return img

In [14]:
input_dir = "./pictures"
output_dir = "./processed"

os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    if filename.lower().endswith((".jpg", ".png", ".jpeg")):
        path = os.path.join(input_dir, filename)

        img = Image.open(set_image_dpi(path))
        processed = preprocess(img)

        cv2.imwrite(os.path.join(output_dir, filename), processed)

### Model config

In [15]:
!pip install -q transformers accelerate bitsandbytes

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00


In [16]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    device_map="auto",
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
)


model.config.use_cache = False


processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

### Text scanning

In [20]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "./processed/IMG_4272.jpg",
            },
            {"type": "text", "text": "You are a document digitization specialist. Return the entire document text in Russian."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['Не стоит прокидаться над ну- тиенсивом, ищущим путь, лучше он прокатится под нас. Однако он прокатится под нас.']


In [18]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "./processed/IMG_4273.jpg",
            },
            {"type": "text", "text": "You are a document digitization specialist. Return the entire document text in Russian."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['Встречи могут быть реализованы в различных форматах, главное — чтобы они предполагали активное вовлечение участников и совместную работу с материалом.']


In [19]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "./processed/IMG_4274.jpg",
            },
            {"type": "text", "text": "You are a document digitization specialist. Return the entire document text in Russian."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['Технологии распознавания печатного текста позволяют быстро обрабатывать и ускорять многие бизнес-процессы.']


In [21]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "./processed/IMG_4275.jpg",
            },
            {"type": "text", "text": "You are a document digitization specialist. Return the entire document text in Russian."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['А/Б - тестирование (сним - тест) + Это метод исследования, при котором сравниваются два варианта (А и Б) веб-страницы, применение или рекламного креатива.']


In [22]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "./processed/IMG_4276.jpg",
            },
            {"type": "text", "text": "You are a document digitization specialist. Return the entire document text in Russian."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['Высказывания Благовестного Августа —\nкак одного из первых христиан-\nских дипосодов, написанное как\nбудто сегодня, а не две тысячи\nлет назад. Слова о любви, жажде,\nсамоотречении, смирении... Завершение\nвнешнего стихия.']


### Model evaluation

In [23]:
!pip install cer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.3 MB/s eta 0:00:00


In [24]:
from cer import calculate_cer_corpus


hyps = ["""Не стоит прокидаться над ну- тиенсивом, ищущим путь, лучше он прокатится под нас. Однако он прокатится под нас.""",
"""Встречи могут быть реализованы в различных форматах, главное — чтобы они предполагали активное вовлечение участников и совместную работу с материалом.""",
"""Технологии распознавания печатного текста позволяют быстро обрабатывать и ускорять многие бизнес-процессы.""",
"""А/Б - тестирование (сним - тест) + Это метод исследования, при котором сравниваются два варианта (А и Б) веб-страницы, применение или рекламного креатива.""",
"""Высказывания Благовестного Августа —\nкак одного из первых христиан-\nских дипосодов, написанное как\nбудто сегодня, а не две тысячи\nлет назад. Слова о любви, жажде,\nсамоотречении, смирении... Завершение\nвнешнего стихия."""]

refs = ["""Не стоит прогибаться под изменчивый мир, пусть лучше он прогнется под нас.
Однажды он прогнется под нас.""",
"""Встречи могут быть реализованы в различных форматах, главное, чтобы они предполагали активное вовлечение участников и совместную работу с материалом.""",
"""Технологии распознавания печатного текста появились около 30 лет назад, существенно облегчив жизнь и ускорив многие бизнес-процессы.""",
"""А/Б-тестирование (сплит-тест) - это метод исследования, при котором сравниваются два варианта (A и B) веб-страницы, приложения или рекламного креатива.""",
"""Высказывания Блаженного Августина, одного из первых христианских философов, написаны как будто сегодня, а не две тысячи лет назад. Слова о любви, жизни, свободе, человеке, смысле... Завораживающее чтение."""]

hyps = [sent.split() for sent in hyps]
refs = [sent.split() for sent in refs]

cer_corpus_score = calculate_cer_corpus(hyps, refs)
cer_corpus_score

{'count': 5,
 'mean': 0.21996852718216078,
 'median': 0.2672811059907834,
 'std': 0.1641519347341989,
 'min': 0.013333333333333334,
 'max': 0.42452830188679247,
 'cer_scores': [0.2972972972972973,
  0.013333333333333334,
  0.42452830188679247,
  0.09740259740259741,
  0.2672811059907834]}